## xBD Dataset Exploration & Preprocessing

Mirrors the structure of `qqb_preprocessing.ipynb`.

Sources:
- `tier3/`  : 12 738 images + GeoJSON labels
- `train/`  : images + GeoJSON labels (targets/ PNG masks are ignored)

Goal: merge both sources, filter to earthquake events only,
map 4-class labels → binary (no-damage → 0 / rest → 1),
and produce `xbd_train.csv` + `xbd_val.csv` ready for training.

In [11]:
import os
import json
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image
from sklearn.model_selection import train_test_split


***0. Root paths***

In [12]:
TIER3_ROOT = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_tier3\tier3"
TRAIN_ROOT = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_train_images_labels_targets\train"

# Each source is defined as (images_dir, labels_dir)
SOURCES = {
    "tier3": (
        os.path.join(TIER3_ROOT, "images"),
        os.path.join(TIER3_ROOT, "labels"),
    ),
    "train": (
        os.path.join(TRAIN_ROOT, "images"),
        os.path.join(TRAIN_ROOT, "labels"),
    ),
}

# Quick sanity check
for source, (img_dir, lbl_dir) in SOURCES.items():
    img_ok = os.path.isdir(img_dir)
    lbl_ok = os.path.isdir(lbl_dir)
    print(f"{source}  images: {'OK' if img_ok else 'NOT FOUND'}  "
          f"labels: {'OK' if lbl_ok else 'NOT FOUND'}")

tier3  images: OK  labels: OK
train  images: OK  labels: OK


***1. File counts per source:***

In [13]:
for source, (img_dir, lbl_dir) in SOURCES.items():
    img_files = [f for f in os.listdir(img_dir) if f.endswith(".png")]
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith(".json")]
    print(f"{source}:  {len(img_files)} images,  {len(lbl_files)} labels")
    print(f"  sample image : {img_files[0]}")
    print(f"  sample label : {lbl_files[0]}")

tier3:  12738 images,  12738 labels
  sample image : joplin-tornado_00000000_post_disaster.png
  sample label : joplin-tornado_00000000_post_disaster.json
train:  5598 images,  5598 labels
  sample image : guatemala-volcano_00000000_post_disaster.png
  sample label : guatemala-volcano_00000000_post_disaster.json


***2. Inspect one sample GeoJSON label to understand the schema:***

In [14]:
def find_first_post_json(labels_dir: str) -> str | None:
    for fname in os.listdir(labels_dir):
        if "post_disaster" in fname and fname.endswith(".json"):
            return os.path.join(labels_dir, fname)
    return None


sample_json = find_first_post_json(SOURCES["tier3"][1])
print("Sample label file:", sample_json)

with open(sample_json, "r") as f:
    data = json.load(f)

print("\nTop-level keys:", list(data.keys()))

if "metadata" in data:
    print("\nMetadata:", data["metadata"])

# Navigate to the feature list
features = data.get("features", {})
if isinstance(features, dict):
    features = features.get("xy", [])

print(f"\nNumber of labelled buildings: {len(features)}")
if features:
    print("First feature keys  :", list(features[0].keys()))
    print("First feature props :", features[0].get("properties", {}))

Sample label file: C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_tier3\tier3\labels\joplin-tornado_00000000_post_disaster.json

Top-level keys: ['features', 'metadata']

Metadata: {'sensor': 'WORLDVIEW02', 'provider_asset_type': 'WORLDVIEW02', 'gsd': 2.35255861282349, 'capture_date': '2011-05-29T17:29:25.433Z', 'off_nadir_angle': 28.4302768707275, 'pan_resolution': 0.585882604122162, 'sun_azimuth': 143.603851318359, 'sun_elevation': 71.8531799316406, 'target_azimuth': 193.251083374023, 'disaster': 'joplin-tornado', 'disaster_type': 'wind', 'catalog_id': '103001000A285500', 'original_width': 1024, 'original_height': 1024, 'width': 1024, 'height': 1024, 'id': 'MjY0Mzk4Mg.vo5dFUhTfoVWnKNLRID8rEkx0A0', 'img_name': 'joplin-tornado_00000000_post_disaster.png'}

Number of labelled buildings: 185
First feature keys  : ['properties', 'wkt']
First feature props : {'feature_type': 'building', 'subtype': 'destroyed', 'uid': 'b6aa615e-56b2-458a-b404-d094b64dcad3'}


***3. Discover all disaster types across both sources:***

In [15]:
def disaster_name(fname: str) -> str:
    """Extract disaster name from filename convention:
       <disaster-name>_<id>_<pre|post>_disaster.png  ->  <disaster-name>
    """
    parts = fname.split("_")
    # Last 3 parts are always: <id> + <pre|post> + disaster.ext
    return "_".join(parts[:-3])


disaster_counts = Counter()

for source, (img_dir, _) in SOURCES.items():
    for fname in os.listdir(img_dir):
        if fname.endswith(".png") and "post_disaster" in fname:
            disaster_counts[disaster_name(fname)] += 1

print(f"Total unique disaster types: {len(disaster_counts)}\n")
print(f"{'Disaster':<40}  {'Post-disaster images'}")
print("-" * 62)
for d, count in sorted(disaster_counts.items()):
    print(f"{d:<40}  {count}")

Total unique disaster types: 19

Disaster                                  Post-disaster images
--------------------------------------------------------------
guatemala-volcano                         18
hurricane-florence                        319
hurricane-harvey                          319
hurricane-matthew                         238
hurricane-michael                         343
joplin-tornado                            149
lower-puna-volcano                        291
mexico-earthquake                         121
midwest-flooding                          279
moore-tornado                             227
nepal-flooding                            619
palu-tsunami                              113
pinery-bushfire                           1845
portugal-wildfire                         1869
santa-rosa-wildfire                       226
socal-fire                                823
sunda-tsunami                             148
tuscaloosa-tornado                        343
woolsey-fire

***4. Filter to earthquake events only:***

Review the disaster list from step 3 and extend `EARTHQUAKE_KEYWORDS` if needed.

In [16]:
# Extend this list after reviewing step 3 output
EARTHQUAKE_KEYWORDS = ["earthquake", "tsunami"]

earthquake_disasters = [
    d for d in disaster_counts
    if any(kw in d.lower() for kw in EARTHQUAKE_KEYWORDS)
]

print(f"Earthquake-related disasters ({len(earthquake_disasters)} found):")
for d in sorted(earthquake_disasters):
    print(f"  {d:<40}  images: {disaster_counts[d]}")

Earthquake-related disasters (3 found):
  mexico-earthquake                         images: 121
  palu-tsunami                              images: 113
  sunda-tsunami                             images: 148


***5. Damage subtype distribution across earthquake events (building level):***

In [17]:
def extract_subtypes(json_path: str) -> list[str]:
    """Return all damage subtype strings from one post-disaster GeoJSON."""
    with open(json_path, "r") as f:
        data = json.load(f)

    features = data.get("features", {})
    if isinstance(features, dict):
        features = features.get("xy", [])
    elif not isinstance(features, list):
        features = []

    return [feat.get("properties", {}).get("subtype", "unknown") for feat in features]


subtype_counter = Counter()

for source, (img_dir, lbl_dir) in SOURCES.items():
    for fname in os.listdir(lbl_dir):
        if "post_disaster" not in fname or not fname.endswith(".json"):
            continue
        if disaster_name(fname) not in earthquake_disasters:
            continue
        subtype_counter.update(extract_subtypes(os.path.join(lbl_dir, fname)))

print("Damage subtype distribution (earthquake events, building level):")
print(f"  {'Subtype':<25}  Count")
print("-" * 40)
for subtype, count in subtype_counter.most_common():
    print(f"  {subtype:<25}  {count}")

Damage subtype distribution (earthquake events, building level):
  Subtype                    Count
----------------------------------------
  no-damage                  71599
  destroyed                  5147
  un-classified              3066
  major-damage               689
  minor-damage               111


***6. Build the per-image DataFrame with binary labels:***

One row per post-disaster image (both sources merged).

Binary mapping:
- `no-damage`      → **0** (intact)
- everything else  → **1** (damaged)

Image-level rule: label = 1 if **any** building in the tile is damaged, else 0.
Tiles with no labelled buildings or only `un-classified` buildings are dropped.

In [18]:
def image_label_from_json(json_path: str) -> int:
    """
    Derive a binary image-level label from a post-disaster GeoJSON.
    Returns  1  if any building is damaged.
    Returns  0  if all buildings are intact.
    Returns -1  if no usable labels found (tile will be dropped).
    """
    subtypes = [
        s for s in extract_subtypes(json_path)
        if s not in ("unknown", "un-classified")
    ]
    if not subtypes:
        return -1
    return 1 if any(s != "no-damage" for s in subtypes) else 0


rows = []

for source, (img_dir, lbl_dir) in SOURCES.items():
    for fname in os.listdir(img_dir):
        if "post_disaster" not in fname or not fname.endswith(".png"):
            continue
        if disaster_name(fname) not in earthquake_disasters:
            continue

        label_fname = fname.replace(".png", ".json")
        label_path  = os.path.join(lbl_dir, label_fname)

        if not os.path.exists(label_path):
            print(f"  [warn] missing label for {fname}")
            continue

        label = image_label_from_json(label_path)
        if label == -1:
            continue

        rows.append({
            "path":     os.path.join(img_dir, fname),
            "label":    label,
            "source":   source,
            "disaster": disaster_name(fname),
        })

df = pd.DataFrame(rows)

print(df.head())
print("\nTotal samples         :", len(df))
print("\nLabel distribution    :")
print(df["label"].value_counts())
print("\nSamples per disaster  :")
print(df["disaster"].value_counts())
print("\nSamples per source    :")
print(df["source"].value_counts())

                                                path  label source  \
0  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      0  tier3   
1  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      0  tier3   
2  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      0  tier3   
3  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1  tier3   
4  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      0  tier3   

        disaster  
0  sunda-tsunami  
1  sunda-tsunami  
2  sunda-tsunami  
3  sunda-tsunami  
4  sunda-tsunami  

Total samples         : 324

Label distribution    :
label
0    197
1    127
Name: count, dtype: int64

Samples per disaster  :
disaster
mexico-earthquake    120
palu-tsunami         111
sunda-tsunami         93
Name: count, dtype: int64

Samples per source    :
source
train    231
tier3     93
Name: count, dtype: int64


***7. Inspect sample images (shape, dtype, value range):***

In [19]:
sample_damaged = df[df["label"] == 1].iloc[0]["path"]
sample_intact  = df[df["label"] == 0].iloc[0]["path"]

for label_name, path in [("damaged", sample_damaged), ("intact", sample_intact)]:
    img = np.array(Image.open(path))
    print(f"\n{label_name.capitalize()} example:")
    print("  Path  :", path)
    print("  Shape :", img.shape)
    print("  dtype :", img.dtype)
    print("  Min   :", img.min())
    print("  Max   :", img.max())


Damaged example:
  Path  : C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_tier3\tier3\images\sunda-tsunami_00000005_post_disaster.png
  Shape : (1024, 1024, 3)
  dtype : uint8
  Min   : 0
  Max   : 255

Intact example:
  Path  : C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\xBD_tier3\tier3\images\sunda-tsunami_00000001_post_disaster.png
  Shape : (1024, 1024, 3)
  dtype : uint8
  Min   : 0
  Max   : 255


***8. Save full DataFrame, shuffle, and stratified split (80/20):***

In [20]:
OUTPUT_DIR = os.path.dirname(os.path.abspath("__file__"))

# Shuffle for consistent ordering, but NO split — the full xBD
# earthquake subset is used as an out-of-domain test set
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

test_csv = os.path.join(OUTPUT_DIR, "xbd_test.csv")
df.to_csv(test_csv, index=False)

print(f"Test set : {len(df)} samples  —  label dist: {dict(df['label'].value_counts())}")
print("Saved to :", test_csv)

Test set : 324 samples  —  label dist: {0: 197, 1: 127}
Saved to : c:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\xbd_dataset\xbd_test.csv
